In [1]:
!python --version
!pip --version


Python 3.13.15
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [3]:
!apt-get update -qq
!apt-get install -y zstd

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (510 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [4]:
!zstd --version

*** Zstandard CLI (64-bit) v1.5.5, by Yann Collet ***


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
!ollama --version

In [7]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("✅ Ollama server started")

✅ Ollama server started


In [8]:
!curl http://127.0.0.1:11434/api/tags

{"models":[]}

In [9]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [10]:
!ollama run llama3.2 "Hello! Introduce yourself in 3 sentences."


I'm an AI designed to provide information and answer questions to the best 
of my knowledge. I'm a large language model, trained on a vast amount of te
text data, which enables me to generate human-like responses. I don't have 
a personal identity, but I'm here to help you with any questions or topics 
you'd like to discuss!



In [11]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 109.0 MB/s eta 0:00:00


In [13]:
%%writefile app.py

import streamlit as st
import requests

# ==========================================
# PAGE CONFIGURATION
# ==========================================

st.set_page_config(
    page_title="Ollama AI Chatbot",
    page_icon="🤖",
    layout="wide"
)


# ==========================================
# CUSTOM CSS
# ==========================================

st.markdown("""
<style>

.stApp {
    background-color: #ffffff;
}

section[data-testid="stSidebar"] {
    background-color: #f7f7f8;
    border-right: 1px solid #e5e5e5;
}

.chat-title {
    text-align: center;
    padding: 25px 0;
}

.chat-title h1 {
    font-size: 32px;
    font-weight: 700;
}

.chat-title p {
    color: #777777;
}

.sidebar-title {
    font-size: 22px;
    font-weight: 700;
}

</style>
""", unsafe_allow_html=True)


# ==========================================
# SESSION STATE
# ==========================================

if "messages" not in st.session_state:
    st.session_state.messages = []


# ==========================================
# SIDEBAR
# ==========================================

with st.sidebar:

    st.markdown(
        '<div class="sidebar-title">🤖 Ollama Chatbot</div>',
        unsafe_allow_html=True
    )

    st.markdown("---")

    if st.button(
        "➕ New Chat",
        use_container_width=True
    ):
        st.session_state.messages = []
        st.rerun()

    st.markdown("### ⚙️ Settings")

    model = st.selectbox(
        "Select Model",
        ["llama3.2"]
    )

    st.markdown("---")

    st.markdown("""
    ### About

    **AI Chatbot**

    Powered by:

    🦙 Ollama
    🧠 Llama 3.2
    ⚡ Streamlit
    """)


# ==========================================
# HEADER
# ==========================================

st.markdown("""
<div class="chat-title">

<h1>🤖 AI Assistant</h1>

<p>
Chat with a free AI model running through Ollama
</p>

</div>
""", unsafe_allow_html=True)


# ==========================================
# CHAT HISTORY
# ==========================================

for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])


# ==========================================
# CHAT INPUT
# ==========================================

prompt = st.chat_input(
    "Message AI Assistant..."
)


# ==========================================
# SEND MESSAGE
# ==========================================

if prompt:

    # Save user message
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    # Display user message
    with st.chat_message("user"):
        st.markdown(prompt)

    # Assistant response
    with st.chat_message("assistant"):

        placeholder = st.empty()

        full_response = ""

        try:

            response = requests.post(
                "http://127.0.0.1:11434/api/chat",

                json={
                    "model": model,

                    "messages":
                    st.session_state.messages,

                    "stream": True
                },

                stream=True,

                timeout=120
            )

            if response.status_code != 200:

                st.error(
                    f"Ollama error: "
                    f"{response.status_code}"
                )

            else:

                for line in response.iter_lines():

                    if line:

                        import json

                        data = json.loads(
                            line.decode("utf-8")
                        )

                        content = (
                            data
                            .get("message", {})
                            .get("content", "")
                        )

                        full_response += content

                        placeholder.markdown(
                            full_response + "▌"
                        )

                placeholder.markdown(
                    full_response
                )

                # Save assistant response
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": full_response
                })

        except Exception as e:

            st.error(
                f"Could not connect to Ollama: {e}"
            )

Overwriting app.py


In [14]:
!ls -la

total 24
drwxr-xr-x 1 root root 4096 Sep 19 13:17 .
drwxr-xr-x 1 root root 4096 Sep 19 13:06 ..
-rw-r--r-- 1 root root 4364 Sep 19 13:17 app.py
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [15]:
%%writefile requirements.txt

streamlit
requests

Writing requirements.txt


In [16]:
%%writefile README.md

# 🤖 Ollama AI Chatbot

A ChatGPT-style AI chatbot built using Streamlit and Ollama.

## Technologies

- Python
- Streamlit
- Ollama
- Llama 3.2

## Features

- ChatGPT-style interface
- Local LLM
- Chat history
- New Chat
- Streaming responses
- No OpenAI API
- No paid API
- No API key required

## Architecture

Google Colab → Ollama → Llama 3.2 → Streamlit

Writing README.md


In [17]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("✅ Cloudflare tunnel installed")

✅ Cloudflare tunnel installed


In [18]:
import subprocess
import time

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("✅ Streamlit started")

✅ Streamlit started


In [19]:
tunnel_process = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("✅ Tunnel started")

✅ Tunnel started


In [20]:
import re
import time

url = None

for _ in range(30):

    line = tunnel_process.stdout.readline()

    if line:

        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:

            url = match.group(0)

            print("\n" + "=" * 60)
            print("🎉 YOUR CHATBOT URL")
            print("=" * 60)
            print(url)
            print("=" * 60)

            break

    time.sleep(1)

2026-09-19T13:19:53Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-19T13:19:53Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-19T13:19:57Z INF +--------------------------------------------------------------------------------------------+
2026-09-19T13:19:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-19T13:19:57Z INF |  https://grey-lot-integrate-technologies.trycloudflare

In [21]:
%%writefile requirements.txt
streamlit
requests

Overwriting requirements.txt


In [22]:
!ls -lh

total 39M
-rw-r--r-- 1 root root 4.3K Sep 19 13:17 app.py
-rwxr-xr-x 1 root root  38M Sep 11 13:43 cloudflared
-rw-r--r-- 1 root root  370 Sep 19 13:18 README.md
-rw-r--r-- 1 root root   19 Sep 19 13:25 requirements.txt
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [23]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 126973 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [24]:
%%writefile app.py
import streamlit as st
import requests
import json
import io
import os
from pathlib import Path

# Optional document libraries
try:
    from pypdf import PdfReader
except:
    PdfReader = None

try:
    from docx import Document
except:
    Document = None

try:
    from PIL import Image
except:
    Image = None

try:
    import pytesseract
except:
    pytesseract = None


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="NOVA AI",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

* {
    font-family: 'Inter', sans-serif;
}

.stApp {
    background:
        radial-gradient(circle at 10% 10%, rgba(99,102,241,0.08), transparent 25%),
        radial-gradient(circle at 90% 20%, rgba(14,165,233,0.08), transparent 25%),
        #ffffff;
}

/* Sidebar */

section[data-testid="stSidebar"] {
    background: linear-gradient(
        180deg,
        #111827 0%,
        #172033 100%
    );
    border-right: 1px solid #263247;
}

section[data-testid="stSidebar"] * {
    color: #f8fafc !important;
}

.sidebar-brand {
    font-size: 25px;
    font-weight: 700;
    padding: 10px 0 20px 0;
}

.sidebar-subtitle {
    color: #94a3b8 !important;
    font-size: 12px;
}

/* Main header */

.hero {
    text-align: center;
    padding: 25px 10px 15px 10px;
}

.hero-icon {
    font-size: 48px;
}

.hero-title {
    font-size: 34px;
    font-weight: 700;
    background: linear-gradient(
        90deg,
        #6366f1,
        #06b6d4
    );
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.hero-subtitle {
    color: #64748b;
    font-size: 14px;
}

/* Chat messages */

[data-testid="stChatMessage"] {
    border-radius: 18px;
    padding: 8px;
}

/* Input */

[data-testid="stChatInput"] {
    border-radius: 20px;
}

/* Buttons */

.stButton > button {
    border-radius: 12px;
    font-weight: 600;
    transition: all 0.2s ease;
}

.stButton > button:hover {
    transform: translateY(-1px);
}

/* Cards */

.info-card {
    background: rgba(248,250,252,0.9);
    border: 1px solid #e2e8f0;
    border-radius: 16px;
    padding: 15px;
    margin-bottom: 10px;
}

.feature-card {
    background: linear-gradient(
        135deg,
        rgba(99,102,241,0.08),
        rgba(6,182,212,0.08)
    );
    border: 1px solid rgba(99,102,241,0.15);
    border-radius: 18px;
    padding: 18px;
}

.status-online {
    color: #16a34a;
    font-weight: 600;
}

.footer {
    text-align: center;
    color: #94a3b8;
    font-size: 11px;
    padding: 20px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# SESSION STATE
# ============================================================

if "messages" not in st.session_state:
    st.session_state.messages = []

if "uploaded_files" not in st.session_state:
    st.session_state.uploaded_files = {}

if "system_prompt" not in st.session_state:
    st.session_state.system_prompt = (
        "You are NOVA AI, a helpful, intelligent and professional AI assistant. "
        "Give clear, accurate and well-structured answers. "
        "Use markdown when useful. "
        "For programming questions, provide clean working code."
    )


# ============================================================
# OLLAMA CONFIG
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"


# ============================================================
# DOCUMENT EXTRACTION
# ============================================================

def extract_pdf(file):
    """Extract text from PDF."""

    if PdfReader is None:
        return "PDF library is not installed."

    try:
        reader = PdfReader(file)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        if not text.strip():
            return (
                "This PDF appears to be scanned/image-based. "
                "Text extraction did not find selectable text."
            )

        return text

    except Exception as e:
        return f"PDF extraction error: {e}"


def extract_docx(file):
    """Extract text from DOCX."""

    if Document is None:
        return "DOCX library is not installed."

    try:
        document = Document(file)

        text = []

        for paragraph in document.paragraphs:
            text.append(paragraph.text)

        return "\n".join(text)

    except Exception as e:
        return f"DOCX extraction error: {e}"


def extract_image(file):
    """OCR image."""

    if Image is None:
        return "Pillow is not installed."

    try:
        image = Image.open(file)

        if pytesseract is None:
            return (
                "Image uploaded successfully, but OCR is not installed."
            )

        text = pytesseract.image_to_string(image)

        if not text.strip():
            return "No readable text detected in this image."

        return text

    except Exception as e:
        return f"Image OCR error: {e}"


def extract_text_file(file):
    """Read text-based files."""

    try:
        return file.read().decode("utf-8", errors="ignore")

    except Exception as e:
        return f"Text extraction error: {e}"


def process_file(uploaded_file):

    filename = uploaded_file.name.lower()

    if filename.endswith(".pdf"):
        return extract_pdf(uploaded_file)

    elif filename.endswith(".docx"):
        return extract_docx(uploaded_file)

    elif filename.endswith((
        ".txt",
        ".md",
        ".csv",
        ".json",
        ".py",
        ".sql",
        ".html",
        ".css"
    )):
        return extract_text_file(uploaded_file)

    elif filename.endswith((
        ".png",
        ".jpg",
        ".jpeg",
        ".webp"
    )):
        return extract_image(uploaded_file)

    else:
        return "This file type is not currently supported."


# ============================================================
# OLLAMA CHAT
# ============================================================

def ask_ollama(messages, model):

    try:

        response = requests.post(
            OLLAMA_URL,
            json={
                "model": model,
                "messages": messages,
                "stream": True
            },
            stream=True,
            timeout=300
        )

        if response.status_code != 200:
            return f"Ollama returned error {response.status_code}"

        full_response = ""

        for line in response.iter_lines():

            if not line:
                continue

            data = json.loads(line.decode("utf-8"))

            content = data.get(
                "message",
                {}
            ).get(
                "content",
                ""
            )

            full_response += content

            yield full_response

    except requests.exceptions.ConnectionError:

        yield (
            "❌ **Cannot connect to Ollama.**\n\n"
            "Please make sure the Ollama server is running:\n\n"
            "`ollama serve`"
        )

    except Exception as e:

        yield f"❌ Error: {e}"


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown(
        '<div class="sidebar-brand">🤖 NOVA AI</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="sidebar-subtitle">'
        'Your intelligent AI workspace'
        '</div>',
        unsafe_allow_html=True
    )

    st.markdown("---")

    # New chat

    if st.button(
        "➕ New Chat",
        use_container_width=True
    ):

        st.session_state.messages = []

        st.rerun()

    st.markdown("### 🧠 AI Model")

    model = st.selectbox(
        "Select Ollama Model",
        [
            "llama3.2"
        ]
    )

    st.markdown("### ⚙️ Settings")

    temperature = st.slider(
        "Creativity",
        min_value=0.0,
        max_value=1.5,
        value=0.7,
        step=0.1
    )

    st.markdown("### 📎 Files")

    uploaded_files = st.file_uploader(
        "Upload documents",
        type=[
            "pdf",
            "png",
            "jpg",
            "jpeg",
            "webp",
            "txt",
            "md",
            "csv",
            "json",
            "docx",
            "py",
            "sql",
            "html",
            "css"
        ],
        accept_multiple_files=True
    )

    if uploaded_files:

        for uploaded_file in uploaded_files:

            if uploaded_file.name not in st.session_state.uploaded_files:

                with st.spinner(
                    f"Processing {uploaded_file.name}..."
                ):

                    extracted = process_file(
                        uploaded_file
                    )

                st.session_state.uploaded_files[
                    uploaded_file.name
                ] = extracted

        st.success(
            f"{len(uploaded_files)} file(s) ready"
        )

    st.markdown("---")

    if st.button(
        "🗑️ Clear Chat",
        use_container_width=True
    ):

        st.session_state.messages = []

        st.rerun()

    st.markdown("---")

    st.markdown(
        """
        <div class="info-card">

        <b>🚀 NOVA AI</b>

        <br><br>

        🦙 Ollama<br>
        🧠 Llama 3.2<br>
        📚 Document AI<br>
        🎤 Voice Input<br>
        ⚡ Streaming

        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# MAIN HEADER
# ============================================================

st.markdown(
    """
    <div class="hero">

        <div class="hero-icon">🤖</div>

        <div class="hero-title">
            NOVA AI
        </div>

        <div class="hero-subtitle">
            Your private AI assistant powered by Ollama
        </div>

    </div>
    """,
    unsafe_allow_html=True
)


# ============================================================
# WELCOME SCREEN
# ============================================================

if not st.session_state.messages:

    st.markdown(
        """
        <div class="feature-card">

        ### 👋 Welcome to NOVA AI

        Ask questions, analyze documents, upload images,
        write code, summarize PDFs or simply have a conversation.

        </div>
        """,
        unsafe_allow_html=True
    )

    st.markdown("### ✨ What can I help with?")

    col1, col2, col3 = st.columns(3)

    with col1:

        st.markdown(
            """
            <div class="info-card">

            📚 <b>Document Analysis</b>

            <br><br>

            Upload PDF, DOCX, TXT or CSV files
            and ask questions about them.

            </div>
            """,
            unsafe_allow_html=True
        )

    with col2:

        st.markdown(
            """
            <div class="info-card">

            👨‍💻 <b>Programming</b>

            <br><br>

            Generate Python, SQL, C, C++,
            HTML and other code.

            </div>
            """,
            unsafe_allow_html=True
        )

    with col3:

        st.markdown(
            """
            <div class="info-card">

            🎤 <b>Voice Assistant</b>

            <br><br>

            Speak your question instead
            of typing it.

            </div>
            """,
            unsafe_allow_html=True
        )


# ============================================================
# DISPLAY CHAT
# ============================================================

for message in st.session_state.messages:

    with st.chat_message(
        message["role"]
    ):

        st.markdown(
            message["content"]
        )


# ============================================================
# VOICE INPUT
# ============================================================

st.markdown("### 🎤 Voice Input")

audio_value = st.audio_input(
    "Record your question"
)

if audio_value:

    st.info(
        "🎤 Audio received. Voice transcription can be connected "
        "with a local Whisper model."
    )


# ============================================================
# TEXT INPUT
# ============================================================

prompt = st.chat_input(
    "Message NOVA AI..."
)


# ============================================================
# PROCESS USER MESSAGE
# ============================================================

if prompt:

    # Add user message

    st.session_state.messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    with st.chat_message("user"):

        st.markdown(prompt)


    # Build context from uploaded files

    file_context = ""

    if st.session_state.uploaded_files:

        file_context += (
            "\n\nThe user has uploaded the following "
            "documents. Use them when relevant:\n\n"
        )

        for filename, text in (
            st.session_state.uploaded_files.items()
        ):

            # Limit context size

            limited_text = text[:12000]

            file_context += (
                f"\n===== {filename} =====\n"
                f"{limited_text}\n"
            )


    # Build AI messages

    ai_messages = [
        {
            "role": "system",
            "content": (
                st.session_state.system_prompt
                + file_context
            )
        }
    ]

    ai_messages.extend(
        st.session_state.messages
    )


    # AI response

    with st.chat_message("assistant"):

        placeholder = st.empty()

        final_response = ""

        for partial_response in ask_ollama(
            ai_messages,
            model
        ):

            final_response = partial_response

            placeholder.markdown(
                final_response + "▌"
            )

        placeholder.markdown(
            final_response
        )


    # Save assistant response

    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": final_response
        }
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    """
    <div class="footer">

    NOVA AI • Powered by Ollama + Llama 3.2 •
    Private Local AI

    </div>
    """,
    unsafe_allow_html=True
)

Overwriting app.py


In [25]:
!pip install -q streamlit requests pypdf python-docx pillow pytesseract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.4 MB/s eta 0:00:00


In [26]:
!pip install -q streamlit requests pypdf python-docx pillow pytesseract

In [27]:
!apt-get update -qq
!apt-get install -y tesseract-ocr

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 93 not upgraded.


In [28]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-09-19T13:16:02.456279416Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [29]:
!pip install -q streamlit

In [30]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [31]:
!cat streamlit.log



2026-09-19 13:28:33.607 Port 8501 is not available


In [32]:
!pkill -f streamlit

In [33]:
!fuser -k 8501/tcp

In [34]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [35]:
!cat streamlit.log



2026-09-19 13:30:23.023 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.110.41.245:8501



In [36]:
%%writefile app.py
import streamlit as st
import requests
import json
from pypdf import PdfReader
from docx import Document
from PIL import Image
import pytesseract

# ============================================================
# CONFIG
# ============================================================

st.set_page_config(
    page_title="NOVA AI",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"

# ============================================================
# CSS
# ============================================================

st.markdown("""
<style>

/* ---------- APP ---------- */

.stApp {
    background: #ffffff;
}

/* ---------- SIDEBAR ---------- */

section[data-testid="stSidebar"] {
    background: #f7f7f8;
    border-right: 1px solid #e5e5e5;
}

section[data-testid="stSidebar"] .block-container {
    padding-top: 1.5rem;
}

.sidebar-logo {
    font-size: 24px;
    font-weight: 700;
    color: #111827;
    margin-bottom: 3px;
}

.sidebar-subtitle {
    color: #6b7280;
    font-size: 12px;
    margin-bottom: 20px;
}

/* ---------- MAIN ---------- */

.main-title {
    text-align: center;
    font-size: 32px;
    font-weight: 700;
    color: #111827;
    margin-top: 25px;
}

.main-subtitle {
    text-align: center;
    color: #6b7280;
    font-size: 14px;
    margin-bottom: 30px;
}

/* ---------- WELCOME ---------- */

.welcome-box {
    max-width: 850px;
    margin: 20px auto;
    padding: 25px;
    border: 1px solid #e5e7eb;
    border-radius: 18px;
    background: #fafafa;
    text-align: center;
}

.welcome-title {
    font-size: 24px;
    font-weight: 600;
    color: #111827;
}

.welcome-text {
    color: #6b7280;
    margin-top: 8px;
}

/* ---------- FEATURE CARDS ---------- */

.feature {
    padding: 18px;
    border: 1px solid #e5e7eb;
    border-radius: 15px;
    background: white;
    min-height: 120px;
}

.feature-icon {
    font-size: 25px;
}

.feature-title {
    font-weight: 600;
    margin-top: 8px;
    color: #111827;
}

.feature-text {
    font-size: 13px;
    color: #6b7280;
}

/* ---------- CHAT ---------- */

[data-testid="stChatMessage"] {
    max-width: 850px;
    margin-left: auto;
    margin-right: auto;
}

/* ---------- INPUT ---------- */

[data-testid="stChatInput"] {
    max-width: 850px;
    margin-left: auto;
    margin-right: auto;
}

/* ---------- BUTTON ---------- */

.stButton > button {
    border-radius: 10px;
    font-weight: 500;
}

/* ---------- FILE ---------- */

[data-testid="stFileUploader"] {
    border-radius: 12px;
}

/* ---------- FOOTER ---------- */

.footer {
    text-align: center;
    color: #9ca3af;
    font-size: 11px;
    margin-top: 30px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# SESSION STATE
# ============================================================

if "messages" not in st.session_state:
    st.session_state.messages = []

if "files" not in st.session_state:
    st.session_state.files = {}


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown(
        "🤖 **NOVA AI**"
    )

    st.caption(
        "Your intelligent AI workspace"
    )

    st.divider()

    if st.button(
        "➕ New chat",
        use_container_width=True
    ):
        st.session_state.messages = []
        st.rerun()

    st.divider()

    st.markdown("### 🧠 Model")

    model = st.selectbox(
        "Ollama model",
        ["llama3.2"],
        label_visibility="collapsed"
    )

    st.divider()

    st.markdown("### 📎 Upload files")

    uploaded_files = st.file_uploader(
        "PDF, DOCX, TXT, CSV, Images",
        type=[
            "pdf",
            "docx",
            "txt",
            "csv",
            "md",
            "json",
            "png",
            "jpg",
            "jpeg",
            "webp"
        ],
        accept_multiple_files=True
    )

    if uploaded_files:

        for file in uploaded_files:

            if file.name not in st.session_state.files:

                filename = file.name.lower()

                try:

                    if filename.endswith(".pdf"):

                        reader = PdfReader(file)

                        text = ""

                        for page in reader.pages:
                            page_text = page.extract_text()

                            if page_text:
                                text += page_text + "\n"

                    elif filename.endswith(".docx"):

                        doc = Document(file)

                        text = "\n".join(
                            p.text
                            for p in doc.paragraphs
                        )

                    elif filename.endswith(
                        (".png", ".jpg", ".jpeg", ".webp")
                    ):

                        image = Image.open(file)

                        text = pytesseract.image_to_string(
                            image
                        )

                    else:

                        text = file.read().decode(
                            "utf-8",
                            errors="ignore"
                        )

                    st.session_state.files[
                        file.name
                    ] = text

                except Exception as e:

                    st.session_state.files[
                        file.name
                    ] = f"File processing error: {e}"

        st.success(
            f"{len(uploaded_files)} file(s) uploaded"
        )

    if st.session_state.files:

        st.markdown("### 📚 Your files")

        for filename in st.session_state.files:

            st.caption(
                f"📄 {filename}"
            )

    st.divider()

    if st.button(
        "🗑️ Clear conversation",
        use_container_width=True
    ):
        st.session_state.messages = []
        st.rerun()

    st.divider()

    st.caption(
        "🦙 Ollama • Llama 3.2"
    )

    st.caption(
        "🔒 Local AI • No API key"
    )


# ============================================================
# MAIN HEADER
# ============================================================

st.markdown(
    '<div class="main-title">🤖 NOVA AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="main-subtitle">'
    'Your private AI assistant powered by Ollama'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# WELCOME SCREEN
# ============================================================

if not st.session_state.messages:

    st.markdown(
        """
        <div class="welcome-box">

        <div class="welcome-title">
        👋 Welcome to NOVA AI
        </div>

        <div class="welcome-text">
        Ask questions, upload documents, analyze images,
        write code, or start a conversation.
        </div>

        </div>
        """,
        unsafe_allow_html=True
    )

    c1, c2, c3 = st.columns(3)

    with c1:
        st.markdown(
            """
            <div class="feature">

            <div class="feature-icon">📚</div>

            <div class="feature-title">
            Document AI
            </div>

            <div class="feature-text">
            Upload PDFs and documents
            and ask questions about them.
            </div>

            </div>
            """,
            unsafe_allow_html=True
        )

    with c2:
        st.markdown(
            """
            <div class="feature">

            <div class="feature-icon">💻</div>

            <div class="feature-title">
            Coding Assistant
            </div>

            <div class="feature-text">
            Generate and explain Python,
            SQL, C, C++ and more.
            </div>

            </div>
            """,
            unsafe_allow_html=True
        )

    with c3:
        st.markdown(
            """
            <div class="feature">

            <div class="feature-icon">🎤</div>

            <div class="feature-title">
            Voice Assistant
            </div>

            <div class="feature-text">
            Record your voice and
            interact with your AI assistant.
            </div>

            </div>
            """,
            unsafe_allow_html=True
        )


# ============================================================
# CHAT HISTORY
# ============================================================

for message in st.session_state.messages:

    with st.chat_message(
        message["role"]
    ):

        st.markdown(
            message["content"]
        )


# ============================================================
# VOICE INPUT
# ============================================================

audio = st.audio_input(
    "🎤 Voice input"
)

if audio:

    st.info(
        "Voice recording received. "
        "Whisper speech-to-text can be connected next."
    )


# ============================================================
# CHAT INPUT
# ============================================================

prompt = st.chat_input(
    "Message NOVA AI..."
)


# ============================================================
# CHAT PROCESSING
# ============================================================

if prompt:

    st.session_state.messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    with st.chat_message("user"):
        st.markdown(prompt)

    # ------------------------------------------
    # FILE CONTEXT
    # ------------------------------------------

    file_context = ""

    if st.session_state.files:

        file_context = (
            "\n\nUploaded file information:\n"
        )

        for filename, content in (
            st.session_state.files.items()
        ):

            file_context += (
                f"\n--- {filename} ---\n"
            )

            file_context += content[:10000]

    # ------------------------------------------
    # OLLAMA MESSAGES
    # ------------------------------------------

    messages = [
        {
            "role": "system",
            "content":
            """
            You are NOVA AI.

            You are a helpful, intelligent AI assistant.

            Give clear and accurate answers.

            Use Markdown for formatting.

            When writing code, provide complete
            and properly formatted code.

            If uploaded documents are provided,
            use them when answering questions.
            """
            + file_context
        }
    ]

    messages.extend(
        st.session_state.messages
    )

    # ------------------------------------------
    # OLLAMA
    # ------------------------------------------

    with st.chat_message("assistant"):

        placeholder = st.empty()

        full_response = ""

        try:

            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": model,
                    "messages": messages,
                    "stream": True
                },
                stream=True,
                timeout=300
            )

            if response.status_code != 200:

                placeholder.error(
                    f"Ollama error: {response.status_code}"
                )

            else:

                for line in response.iter_lines():

                    if not line:
                        continue

                    data = json.loads(
                        line.decode("utf-8")
                    )

                    content = data.get(
                        "message",
                        {}
                    ).get(
                        "content",
                        ""
                    )

                    full_response += content

                    placeholder.markdown(
                        full_response + "▌"
                    )

                placeholder.markdown(
                    full_response
                )

                st.session_state.messages.append(
                    {
                        "role": "assistant",
                        "content": full_response
                    }
                )

        except requests.exceptions.ConnectionError:

            placeholder.error(
                "❌ Ollama is not running. "
                "Start Ollama with `ollama serve`."
            )

        except Exception as e:

            placeholder.error(
                f"❌ Error: {e}"
            )


# ============================================================
# FOOTER
# ============================================================

st.markdown(
    '<div class="footer">'
    'NOVA AI • Ollama + Llama 3.2 • Local AI'
    '</div>',
    unsafe_allow_html=True
)

Overwriting app.py


In [37]:
!ls -lah

total 57M
drwxr-xr-x 1 root root 4.0K Sep 19 13:28 .
drwxr-xr-x 1 root root 4.0K Sep 19 13:06 ..
-rw-r--r-- 1 root root  13K Sep 19 13:33 app.py
-rwxr-xr-x 1 root root  38M Sep 11 13:43 cloudflared
-rw-r--r-- 1 root root  19M Sep 11 13:43 cloudflared-linux-amd64.deb
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
-rw-r--r-- 1 root root  370 Sep 19 13:18 README.md
-rw-r--r-- 1 root root   19 Sep 19 13:25 requirements.txt
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data
-rw-r--r-- 1 root root  323 Sep 19 13:30 streamlit.log


In [ ]:
%%writefile README.md
# 🤖 NOVA AI

A modern ChatGPT-style AI chatbot powered by Ollama and Llama 3.2.

## Features

- ChatGPT-style interface
- Ollama Llama 3.2
- Streaming responses
- PDF upload
- DOCX upload
- TXT/CSV/JSON upload
- Image OCR
- Voice input interface
- Chat history
- New Chat
- Clear conversation
- Light/Dark Streamlit theme

## Technologies

- Python
- Streamlit
- Ollama
- Llama 3.2
- PyPDF
- python-docx
- Tesseract OCR

## Architecture

Google Colab → Streamlit → Ollama → Llama 3.2

## Important

Ollama must be running for the chatbot to generate responses.

In [38]:
%%writefile .gitignore
__pycache__/
*.pyc
.env
.env.*
.streamlit/secrets.toml

*.pt
*.pth
*.bin
*.safetensors

uploads/
temp/
output/

cloudflared-linux-amd64.deb
*.log

sample_data/

Writing .gitignore


In [39]:
!ls -lah

total 57M
drwxr-xr-x 1 root root 4.0K Sep 19 13:36 .
drwxr-xr-x 1 root root 4.0K Sep 19 13:06 ..
-rw-r--r-- 1 root root  13K Sep 19 13:33 app.py
-rwxr-xr-x 1 root root  38M Sep 11 13:43 cloudflared
-rw-r--r-- 1 root root  19M Sep 11 13:43 cloudflared-linux-amd64.deb
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
-rw-r--r-- 1 root root  160 Sep 19 13:36 .gitignore
-rw-r--r-- 1 root root  370 Sep 19 13:18 README.md
-rw-r--r-- 1 root root   19 Sep 19 13:25 requirements.txt
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data
-rw-r--r-- 1 root root  323 Sep 19 13:30 streamlit.log


In [40]:
!rm -f NOVA_AI.zip
!zip NOVA_AI.zip app.py requirements.txt README.md .gitignore

  adding: app.py (deflated 75%)
  adding: requirements.txt (stored 0%)
  adding: README.md (deflated 37%)
  adding: .gitignore (deflated 24%)


In [41]:
from google.colab import files

files.download("NOVA_AI.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>